# Phase 1: nonlinear DC motor simulation

Validate the permanent-magnet DC motor plant before adding faults, learned dynamics, reliability logic, or control.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

src = Path.cwd() / 'src'
if not src.exists():
    src = Path.cwd().parent / 'src'
sys.path.insert(0, str(src))

from motor_model import DCMotorParams, simulate_motor

## Nominal parameters and plotting helper

In [2]:
params = DCMotorParams()
params

DCMotorParams(resistance=2.0, inductance=0.5, back_emf_constant=0.1, torque_constant=0.1, inertia=0.01, viscous_friction=0.002, coulomb_friction=0.02, friction_smoothing_speed=0.1)

In [3]:
def plot_experiment(title, time, states, voltage, load_torque):
    current, speed = states.T
    voltage_trace = np.array([voltage(t) for t in time])
    load_trace = np.array([load_torque(t) for t in time])
    rpm = speed * 60 / (2 * np.pi)

    fig, axes = plt.subplots(4, 1, sharex=True, figsize=(9, 8))
    fig.suptitle(title)
    axes[0].plot(time, voltage_trace)
    axes[0].set_ylabel('Voltage (V)')
    axes[1].plot(time, current)
    axes[1].set_ylabel('Current (A)')
    axes[2].plot(time, speed, color='tab:blue')
    axes[2].set_ylabel('Speed (rad/s)', color='tab:blue')
    rpm_axis = axes[2].twinx()
    rpm_axis.plot(time, rpm, color='tab:orange', alpha=0.8)
    rpm_axis.set_ylabel('Speed (RPM)', color='tab:orange')
    axes[3].plot(time, load_trace)
    axes[3].set_ylabel('Load torque (N m)')
    axes[3].set_xlabel('Time (s)')
    for axis in axes:
        axis.grid(alpha=0.3)
    fig.tight_layout()
    plt.show()

## Experiment 1: constant voltage step response

In [4]:
step_voltage = lambda t: 12.0 if t >= 0.5 else 0.0
zero_load = lambda t: 0.0

step_time, step_states = simulate_motor(
    step_voltage, zero_load, simulation_time=8.0, timestep=0.01, params=params
)
plot_experiment(
    '12 V step response', step_time, step_states, step_voltage, zero_load
)

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_9836\3471546509.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Experiment 2: changing voltage input

In [5]:
def changing_voltage(t):
    if t < 0.5:
        return 0.0
    if t < 3.0:
        return 6.0
    if t < 5.5:
        return 12.0
    return 8.0


changing_time, changing_states = simulate_motor(
    changing_voltage, zero_load, simulation_time=8.0, timestep=0.01, params=params
)
plot_experiment(
    'Changing voltage input',
    changing_time,
    changing_states,
    changing_voltage,
    zero_load,
)

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_9836\3471546509.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Experiment 3: sudden load torque disturbance

In [6]:
disturbance_voltage = lambda t: 12.0 if t >= 0.5 else 0.0
sudden_load = lambda t: 0.08 if t >= 6.0 else 0.0

disturbance_time, disturbance_states = simulate_motor(
    disturbance_voltage,
    sudden_load,
    simulation_time=12.0,
    timestep=0.01,
    params=params,
)
plot_experiment(
    'Sudden load torque disturbance',
    disturbance_time,
    disturbance_states,
    disturbance_voltage,
    sudden_load,
)

C:\Users\Lakshya\AppData\Local\Temp\ipykernel_9836\3471546509.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sanity checks

In [7]:
_, zero_voltage_states = simulate_motor(
    lambda t: 0.0,
    simulation_time=4.0,
    timestep=0.01,
    params=params,
    initial_state=(0.0, 20.0),
)
_, low_voltage_states = simulate_motor(
    lambda t: 6.0, simulation_time=10.0, timestep=0.01, params=params
)
_, high_voltage_states = simulate_motor(
    lambda t: 12.0, simulation_time=10.0, timestep=0.01, params=params
)

speed_before_load = disturbance_states[
    (disturbance_time >= 5.5) & (disturbance_time < 6.0), 1
].mean()
speed_after_load = disturbance_states[disturbance_time >= 11.5, 1].mean()

assert zero_voltage_states[:, 1].max() <= 20.0 + 1e-9
assert high_voltage_states[-1, 1] > low_voltage_states[-1, 1]
assert speed_after_load < speed_before_load
print('Sanity checks passed: passive coast-down, voltage ordering, load response.')

Sanity checks passed: passive coast-down, voltage ordering, load response.


## Phase 1 summary

The two-state permanent-magnet DC motor model is

$$\frac{di}{dt}=\frac{V-Ri-K_b\omega}{L}, \qquad \frac{d\omega}{dt}=\frac{K_ti-B\omega-T_{load}-T_{friction}}{J}.$$

Nominal SI parameters: $R=2.0$, $L=0.5$, $K_b=0.1$, $K_t=0.1$, $J=0.01$, and $B=0.002$. The nonlinear friction is smooth Coulomb friction, $T_{friction}=0.02\tanh(\omega/0.1)$ N m.

The 12 V step produces a current transient followed by a smooth speed rise. With changing voltage, speed follows each new operating level with electrical and mechanical lag. Applying a 0.08 N m load while holding voltage fixed causes the speed to fall to a lower operating point. The sanity checks also confirm passive coast-down at zero voltage, higher steady speed at higher voltage, and speed reduction under added load. No dataset is saved in this phase.